In [1]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# Old-age dependency ratio: population at/above SPA per 100 working-age adults (16–SPA).
# Source: OBR July 2026 FRS, Chart 2.4 (exact annual data).
years = list(range(1991, 2076))
oadr = [30.0,30.1,30.1,30.1,30.1,30.0,30.0,30.0,29.9,29.9,29.8,29.8,29.8,29.9,29.9,
29.9,30.2,30.7,31.1,31.2,30.9,31.0,31.0,31.0,30.6,30.1,29.6,29.0,28.3,27.6,27.6,27.8,
27.8,28.0,28.2,27.8,26.8,27.2,27.7,28.2,29.1,29.9,30.4,31.0,31.5,32.0,32.2,31.9,31.2,
31.3,31.5,31.7,31.9,31.9,32.0,32.1,32.3,32.6,32.9,33.2,33.5,33.8,34.2,34.6,35.0,35.5,
35.9,36.4,36.9,37.3,37.8,38.2,38.6,39.0,39.5,39.9,40.3,40.7,40.9,41.2,41.5,41.9,42.0,
41.5,40.5]
df = pd.DataFrame({"year": years, "oadr": oadr})
df["date"] = pd.to_datetime(df["year"], format="%Y")
df["seg"] = df["year"].apply(lambda y: "projection" if y >= 2025 else "historical")
df["series"] = "x"

base = alt.Chart(df).encode(
    x=alt.X("date:T", axis=alt.Axis(format="%Y", tickCount=8), title=None),
    y=alt.Y("oadr:Q", scale=alt.Scale(domain=[0, 45]),
            title="Old-age dependency ratio (%)",
            axis=alt.Axis(labelExpr="datum.label + '%'")),
    color=alt.Color("series:N", legend=None))

hist = base.transform_filter("datum.year<=2025").mark_line(strokeWidth=2.4)
proj = base.transform_filter("datum.year>=2025").mark_line(strokeWidth=2.4, strokeDash=[5,3])

now = alt.Chart(pd.DataFrame({"date":pd.to_datetime(["2025-01-01"])})).mark_rule(
    color="#94a3b8", strokeDash=[2,3]).encode(x="date:T")
now_txt = alt.Chart(pd.DataFrame({"date":pd.to_datetime(["2025-01-01"]),"t":["Today"]})).mark_text(
    align="left", dx=4, dy=-4, fontSize=10, color="#64748b").encode(
    x="date:T", y=alt.value(8), text="t:N")

caption = alt.Title(
    text="Source: OBR, July 2026 Fiscal Risks and Sustainability report (Chart 2.4)",
    subtitle=[
        "Population at/above state pension age per 100 working-age adults. Dashed = projection.",
    ],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (now + now_txt + hist + proj)
    .properties(width=700, height=270, title=caption)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="dependency_ratio", svg=True)
chart.save("dependency_ratio.png", scale_factor=2.0)
chart

alt.LayerChart(...)